# **`Inatel - C24 (Inteligência Artificial) - 2026/1`**

# <font color='green'>**Atividade 08: RNN**</font>

## <font color='#2D9CDB'>**LEIA ATENTAMENTE AS INSTRUÇÕES A SEGUIR**</font>
- Importe este notebook no [Google Colab](https://colab.research.google.com/) para resolver os exercícios;
- Consulte o material disponibilizado pelo Prof. Felipe Figueiredo para revisar os conceitos;
- Utilize os recursos disponíveis na Internet (documentações e artigos científicos) para complementar seus estudos;
- <font color='red'>**A ATIVIDADE DEVERÁ SER REALIZADA COM O [MONITOR](mailto:matheus.botelho@ges.inatel.br) EM UM DOS SEGUINTES HORÁRIOS:**</font>

| Monitor                       | Dia           | Hora                               | Local           |
|-------------------------------|---------------|------------------------------------|---------------- |
| Matheus Botelho Sampaio Netto | Terça-feira   | <font color='orange'>17:30</font>  | 1.4 (prédio VI) |
| Matheus Botelho Sampaio Netto | Quinta-feira  | <font color='orange'>17:30</font>  | 1.4 (prédio VI) |
| Matheus Botelho Sampaio Netto | Sábado        | <font color='#2D9CDB'>10:00</font> | Teams (ao vivo) |
| Matheus Botelho Sampaio Netto | Sábado        | <font color='#2D9CDB'>13:30</font> | Teams (ao vivo) |

- <font color='red'>**NÃO**</font> remova as células de Código já presentes neste notebook;
- <font color='red'>**NÃO**</font> modifique as células de Markdown (em <font color='green'>verde</font> ou <font color='#2D9CDB'>azul</font>) presentes neste notebook;
- Após cada questão, há uma célula para você implementar e responder a questão;
- É permitido adicionar mais células (de código ou markdown) antes da próxima pergunta;
- Caso precise utilizar bibliotecas que não estão instaladas nativamente no Colab, inclua uma célula de código com o comando de instalação.</font>
  - Exemplo: `!pip install nome_da_biblioteca`
- <font color='red'>**Renomeie os termos `LL` para `sua_turma_de_laboratorio` e `MMMM` para `seu_numero_de_matricula` no nome do arquivo.**</font>
  - Exemplo: `C24_2026_1_L1_Atividade_08_1234.ipynb`)
- <font color='magenta'>**Faça download do notebook com a resolução no Google Colab, mantendo a saída de todas as células, e anexe-o à tarefa do Teams.**</font>

# <font color='green'><u><b>Preparação</b></u></font>

In [1]:
!pip install tensorflow numpy pandas matplotlib seaborn scikit-learn

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf
import os

from sklearn.preprocessing import MinMaxScaler

tf.random.set_seed(42)
np.random.seed(42)

# <font color='green'><u><b>Parte 1 - Conjunto de Dados</b></u></font>

Não altere o conteúdo da célula a seguir!

In [3]:
zip_path = tf.keras.utils.get_file(
    fname="jena_climate_2009_2016.csv.zip",
    origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
    cache_dir=os.getcwd(),
    extract=True
)

csv_path = os.path.join(os.path.splitext(zip_path)[0], 'jena_climate_2009_2016.csv')

target_col = "T (degC)"

temperature = pd.read_csv(csv_path)[target_col].values.reshape(-1, 1)

temperature

13568290/13568290 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


array([[-8.02],
       [-8.41],
       [-8.51],
       ...,
       [-3.16],
       [-4.23],
       [-4.82]])

### <font color='#2D9CDB'>Q1) O dataset [Jena Climate](https://www.kaggle.com/datasets/mnassrib/jena-climate) é amplamente utilizado no estudo de redes neurais recorrentes (RNNs), LSTMs e previsão de séries temporais. Pesquise brevemente sobre o dataset e descreva, em um único parágrafo, o domínio de aplicação da base de dados, o objetivo para o qual ela foi criada, o período de coleta dos dados, os tipos de variáveis meteorológicas registradas e por que esse conjunto de dados é adequado para problemas de previsão temporal utilizando redes neurais recorrentes.</font>

O dataset tem como objetivo é fornecer medições climáticas reais para estudos de modelagem e previsão de séries temporais utilizando técnicas de aprendizado de máquina. Os dados foram coletados entre os anos de 2009 e 2016, com registros realizados a cada 10 minutos. A base contém variáveis meteorológicas, como temperatura, pressão atmosférica, umidade relativa, velocidade e direção do vento, densidade do ar e radiação. Esse conjunto de dados é amplamente utilizado em pesquisas envolvendo RNNs e LSTMs porque apresenta dependências temporais, sazonalidades e padrões climáticos reais, tornando-o adequado para problemas de previsão de séries temporais.

### <font color='#2D9CDB'>Q2) Utilizando a variável <code>temperature</code>, plote a série temporal completa de temperaturas registradas no dataset. Em seguida, informe a quantidade total de amostras, a temperatura mínima, a temperatura máxima, a temperatura média e o desvio padrão da série. Com base no gráfico obtido, descreva brevemente os padrões observados, comentando sobre possíveis tendências, sazonalidades e variações ao longo do tempo.</font>

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(temperature)
plt.title('Série Temporal de Temperatura')
plt.xlabel('Amostra')
plt.ylabel('Temperatura (°C)')
plt.grid()
plt.show()

print("Quantidade de amostras:", len(temperature))
print("Temperatura mínima:", temperature.min())
print("Temperatura máxima:", temperature.max())
print("Temperatura média:", temperature.mean())
print("Desvio padrão:", temperature.std())

Pode ser observada uma forte sazonalidade anual, com ciclos de aumento e redução de temperatura correspondentes às estações do ano. Não tem tendência crescente ou decrescente significativa ao longo do período analisado, mas existem oscilações de curto prazo associadas às condições climáticas diárias. A série apresenta comportamento periódico.

### <font color='#2D9CDB'>Q3) Divida a série temporal em três subconjuntos: treinamento (60%), validação (20%) e teste (20%), preservando a ordem temporal das amostras. Informe a quantidade de amostras em cada subconjunto e explique por que, em problemas de previsão temporal, não é adequado embaralhar os dados antes da divisão.</font>

In [ ]:
n = len(temperature)

train_size = int(0.6 * n)
val_size = int(0.2 * n)
test_size = n - train_size - val_size

train = temperature[:train_size]
val = temperature[train_size:train_size+val_size]
test = temperature[train_size+val_size:]

print("Treinamento:", len(train))
print("Validação:", len(val))
print("Teste:", len(test))

Em séries temporais não devemos embaralhar os dados porque a ordem cronológica contém informações importantes sobre a evolução do fenômeno. O embaralhamento causaria com que o modelo tivesse acesso indireto a informações futuras durante o treinamento, causando vazamento de dados e produzindo avaliações irreais do desempenho.

### <font color='#2D9CDB'>Q4) Normalize os conjuntos de treinamento, validação e teste utilizando a técnica Min-Max Scaling, de forma que os valores fiquem no intervalo [0, 1]. Apresente os valores mínimo e máximo do conjunto de treinamento antes e depois da normalização e explique brevemente por que a normalização é importante para o treinamento de redes neurais.</font>

In [ ]:
scaler = MinMaxScaler()

train_norm = scaler.fit_transform(train)

val_norm = scaler.transform(val)
test_norm = scaler.transform(test)

print("Antes da normalização")
print("Min:", train.min())
print("Max:", train.max())

print("\nDepois da normalização")
print("Min:", train_norm.min())
print("Max:", train_norm.max())

A normalização reduz diferenças de escala entre os dados, facilita o processo de otimização, melhora a estabilidade numérica e acelera a convergência do treinamento das redes neurais.

# <font color='green'><u><b>Parte 2 - Preparação das Séries Temporais</b></u></font>

### <font color='#2D9CDB'>Q5) Para treinar uma LSTM, é necessário organizar a série temporal em sequências de entrada e saídas desejadas. Implemente uma função chamada <code>create_sequences()</code> que utilize uma janela temporal de 30 amostras, de forma que cada entrada seja composta pelas 30 temperaturas anteriores e a saída corresponda à temperatura seguinte. Gere os conjuntos de treinamento, validação e teste e informe as dimensões de <code>X_train</code> e <code>y_train</code>. Explique brevemente o significado de cada dimensão.</font>

In [ ]:
def create_sequences(data, window_size=30):
    X = []
    y = []

    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])

    return np.array(X), np.array(y)

window_size = 30

X_train, y_train = create_sequences(train_norm, window_size)
X_val, y_val = create_sequences(val_norm, window_size)
X_test, y_test = create_sequences(test_norm, window_size)

print(X_train.shape)
print(y_train.shape)

A janela temporal define quantas observações passadas serão utilizadas para prever o próximo valor da série. Nesse caso, a rede recebe as 30 temperaturas anteriores e tenta estimar a temperatura seguinte.

### <font color='#2D9CDB'>Q6) Exiba graficamente a primeira sequência presente em <code>X_train</code> e destaque o valor correspondente em <code>y_train</code>. Explique como essa sequência será utilizada pela LSTM para realizar a previsão da próxima temperatura.</font>

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(X_train[0], label='Sequência')
plt.scatter(
    len(X_train[0]),
    y_train[0],
    color='red',
    label='Próxima temperatura'
)

plt.legend()
plt.grid()
plt.show()

A LSTM recebe a sequência completa de 30 temperaturas e processa cada valor em ordem temporal. Ao final da sequência, o estado interno da rede contém informações relevantes do histórico observado e é utilizado para prever a próxima temperatura.

# <font color='green'><u><b>Parte 3 - Construindo a Primeira LSTM</b></u></font>

### <font color='#2D9CDB'>Q7) Construa uma rede neural recorrente composta por uma camada LSTM com uma unidade de memória e uma camada de saída totalmente conectada com um neurônio. Utilize a função de perda MSE (<code>mean_squared_error</code>) e o otimizador Adam. Apresente o resumo da arquitetura gerado pelo método <code>summary()</code> e informe o número total de parâmetros treináveis do modelo.</font>

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(1, input_shape=(30,1)),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

model.summary()

14 parâmetros treináveis

### <font color='#2D9CDB'>Q8) Treine o modelo por 10 épocas utilizando os conjuntos de treinamento e validação. Em seguida, plote as curvas de perda de treinamento e validação ao longo das épocas. Com base nos gráficos obtidos, descreva brevemente o comportamento do treinamento e indique se há evidências de sobreajuste (overfitting) ou problemas de generalização.</font>

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_val,y_val)
)

plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')

plt.legend()
plt.grid()
plt.show()

Espera-se que as perdas de treinamento e validação diminuam ao longo das épocas. Caso ambas sigam próximas, o modelo apresenta boa capacidade de generalização. Se a perda de treinamento continuar diminuindo enquanto a perda de validação aumenta, há evidências de overfitting.

### <font color='#2D9CDB'>Q9) Avalie o modelo utilizando o conjunto de teste e reporte o valor do erro quadrático médio (MSE). Em seguida, explique brevemente o que essa métrica representa no contexto do problema de previsão de temperatura.</font>

In [ ]:
mse = model.evaluate(X_test, y_test)
print("MSE:", mse)

O MSE (Mean Squared Error) representa a média dos quadrados dos erros entre os valores previstos e os valores reais. Quanto menor o MSE, melhor a capacidade preditiva do modelo.

### <font color='#2D9CDB'>Q10) Utilize o modelo treinado para gerar previsões no conjunto de teste. Plote em um mesmo gráfico os valores reais e previstos para as 100 primeiras amostras do conjunto de teste e descreva brevemente a qualidade das previsões obtidas.</font>

In [ ]:
pred = model.predict(X_test)

plt.figure(figsize=(12,5))

plt.plot(y_test[:100], label='Real')
plt.plot(pred[:100], label='Previsto')

plt.legend()
plt.grid()
plt.show()

O gráfico permite verificar visualmente o quanto as previsões acompanham a série real. Quanto mais próximas estiverem as curvas, melhor será o desempenho do modelo.

# <font color='green'><u><b>Parte 4 - Comparação de Arquiteturas</b></u></font>

### <font color='#2D9CDB'>Q11) Modifique a arquitetura da rede para utilizar 10 unidades na camada LSTM. Apresente o resumo da arquitetura gerado pelo método <code>summary()</code> e informe o número total de parâmetros treináveis do modelo. Em seguida, treine a rede por 10 épocas, avalie o modelo no conjunto de teste e reporte o valor de MSE obtido.</font>

In [ ]:
model10 = tf.keras.Sequential([
    tf.keras.layers.LSTM(10, input_shape=(30,1)),
    tf.keras.layers.Dense(1)
])

model10.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

model10.summary()

In [ ]:
history10 = model10.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_val,y_val)
)

mse10 = model10.evaluate(X_test,y_test)

print(mse10)

491 parâmetros treináveis

### <font color='#2D9CDB'>Q12) Repita o experimento utilizando 20 unidades na camada LSTM. Apresente o resumo da arquitetura gerado pelo método <code>summary()</code> e informe o número total de parâmetros treináveis do modelo. Em seguida, treine a rede por 10 épocas, avalie o modelo no conjunto de teste e reporte o valor de MSE obtido.</font>

In [ ]:
model20 = tf.keras.Sequential([
    tf.keras.layers.LSTM(20, input_shape=(30,1)),
    tf.keras.layers.Dense(1)
])

model20.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

model20.summary()

In [ ]:
history20 = model20.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_val,y_val)
)

mse20 = model20.evaluate(X_test,y_test)

print(mse20)

1781 parâmetros treináveis

### <font color='#2D9CDB'>Q13) Compare os modelos com 1, 10 e 20 unidades na camada LSTM em termos de número de parâmetros treináveis e desempenho no conjunto de teste. Organize os resultados em uma tabela contendo a quantidade de unidades LSTM, o número de parâmetros treináveis e o valor de MSE. Com base nos resultados obtidos, discuta brevemente a relação entre a complexidade do modelo e sua capacidade preditiva.</font>

In [ ]:
resultados = pd.DataFrame({
    "Unidades LSTM":[1,10,20],
    "Parâmetros":[14,491,1781],
    "MSE":[mse,mse10,mse20]
})

resultados

O aumento da quantidade de unidades LSTM aumenta significativamente o número de parâmetros treináveis. Em geral, modelos mais complexos conseguem capturar padrões mais sofisticados, porém também apresentam maior custo computacional e risco de overfitting.

# <font color='green'><u><b>Parte 5 - Influência da Janela Temporal</b></u></font>

### <font color='#2D9CDB'>Q14) Repita o experimento utilizando uma janela temporal de 15 amostras e uma camada LSTM com 10 unidades. Treine o modelo por 10 épocas, avalie o modelo no conjunto de teste e reporte o valor de MSE obtido.</font>

In [ ]:
window_size = 15

X_train15, y_train15 = create_sequences(train_norm,15)
X_val15, y_val15 = create_sequences(val_norm,15)
X_test15, y_test15 = create_sequences(test_norm,15)

model15 = tf.keras.Sequential([
    tf.keras.layers.LSTM(10, input_shape=(15,1)),
    tf.keras.layers.Dense(1)
])

model15.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

history15 = model15.fit(
    X_train15,
    y_train15,
    epochs=10,
    validation_data=(X_val15,y_val15)
)

mse15 = model15.evaluate(X_test15,y_test15)

print(mse15)

### <font color='#2D9CDB'>Q15) Repita o experimento utilizando uma janela temporal de 60 amostras e uma camada LSTM com 10 unidades. Treine o modelo por 10 épocas, avalie o modelo no conjunto de teste e reporte o valor de MSE obtido.</font>

In [ ]:
window_size = 60

X_train60, y_train60 = create_sequences(train_norm,60)
X_val60, y_val60 = create_sequences(val_norm,60)
X_test60, y_test60 = create_sequences(test_norm,60)

model60 = tf.keras.Sequential([
    tf.keras.layers.LSTM(10, input_shape=(60,1)),
    tf.keras.layers.Dense(1)
])

model60.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

history60 = model60.fit(
    X_train60,
    y_train60,
    epochs=10,
    validation_data=(X_val60,y_val60)
)

mse60 = model60.evaluate(X_test60,y_test60)

print(mse60)

### <font color='#2D9CDB'>Q16) Compare os resultados obtidos para janelas temporais de 15, 30 e 60 amostras utilizando uma camada LSTM com 10 unidades. Organize os resultados em uma tabela contendo o tamanho da janela temporal, o valor de MSE e uma estimativa do tempo médio de treinamento por época (dica: utilize os valores exibidos nos logs de treinamento). Com base nos resultados, discuta brevemente o impacto do tamanho da janela temporal no desempenho e no custo computacional do modelo.</font>

In [ ]:
janela_df = pd.DataFrame({
    "Janela":[15,30,60],
    "MSE":[mse15,mse10,mse60],
    "Tempo por época":["logs","logs","logs"]
})

janela_df

Janelas maiores fornecem mais contexto histórico para a rede, mas aumentam o custo computacional e o tempo de treinamento. Janelas muito pequenas podem não capturar adequadamente padrões temporais importantes. O melhor tamanho depende do equilíbrio entre desempenho e eficiência computacional.

# <font color='green'><u><b>Parte 6 - Análise Final</b></u></font>

### <font color='#2D9CDB'>Q17) Com base em todos os experimentos realizados nesta atividade, discuta os fatores que mais influenciaram o desempenho do modelo. Considere o impacto da quantidade de unidades LSTM, do tamanho da janela temporal e do custo computacional. Em sua opinião, qual configuração apresentou o melhor equilíbrio entre precisão e eficiência? Justifique sua resposta.</font>

Os fatores que mais influenciaram o desempenho foram a quantidade de unidades LSTM e o tamanho da janela temporal. O aumento do número de unidades permitiu maior capacidade de aprendizado e também elevou o número de parâmetros e o custo computacional. Da mesma forma, as janelas maiores forneceram mais contexto histórico para a previsão, porém aumentaram o tempo de treinamento.
A configuração com 10 unidades LSTM e janela de 30 amostras tende a apresentar um bom equilíbrio entre precisão, capacidade de generalização e eficiência computacional, evitando tanto a limitação de modelos muito simples quanto o custo excessivo de arquiteturas maiores.